```
---
title: Replication Plan for Empirical Asset Pricing via Machine Learning
authors:
  - name: Saahit Karumuri
    email: saahitk2@illinois.edu
    department: Financial Engineering
    affiliation: University of Illinois
abstract: |
  This notebook prepares a replication of Gu, Kelly, and Xiu (2020), Empirical Asset Pricing via Machine Learning. The original paper studies expected stock excess returns as a supervised prediction problem and compares linear, regularized linear, dimension-reduction, tree-based, and neural-network models on a large U.S. equity panel. The planned replication reconstructs the monthly stock-level panel, applies the paper's cross-sectional characteristic ranking and macro-characteristic interaction design, preserves temporal ordering in train, validation, and test samples, evaluates forecasts with out-of-sample R2 against a zero benchmark, and studies portfolio performance through forecast-sorted deciles. Because the restricted CRSP/Compustat-style source data are not included in this repository, the notebook documents exact data requirements and provides executable scaffolding for the replication pipeline.
```



# Introduction

Gu, Kelly, and Xiu study one of the central empirical asset-pricing problems: measuring conditional expected stock returns, or risk premiums. Their framing is deliberately predictive. For stock $i$ in month $t$, the object of interest is

$$
E_t(r_{i,t+1}) = g^*(z_{i,t}),
$$

where $r_{i,t+1}$ is the next-month excess return and $z_{i,t}$ is the vector of information known at time $t$. The paper asks which statistical method best approximates the unknown function $g^*(\cdot)$ when the predictor set is large, noisy, correlated, and potentially nonlinear.

The replication target is not a single trading rule. It build a large monthly stock panel, fit several model families under strict time-ordered validation, compare out-of-sample forecasts, and translate forecasts into economically interpretable portfolios. The original paper's main conclusion is that regularization is essential and that nonlinear interaction models, especially trees and neural networks, produce the strongest stock-level and portfolio-level forecasts.


# Paper Summary

<!-- Start with a single paragraph in précis form. -->
<!-- See @PetersonReplication p. 1-2 for details. -->
<!-- Complete this section with paragraphs describing each major point in the paper. -->
<!-- The entire summary will be 4-10 paragraphs. -->

The paper's main insight is that the expected excess return is the conditional expectation of a future realized excess return. This makes asset pricing a natural domain for machine learning, it is a difficult one because stock returns have low signal tonoise ratios, predictors are numerous and correlated, and overfitting can easily dominate apparent in-sample success.

The authors compare a broad model set: elastic net, principal components regression, partial least squares, generalized linear models with spline expansions and group lasso, random forests, gradient boosted regression trees, and feed-forward neural networks with one to five hidden layers.

Full OLS with all predictors performs poorly because it overfits. Penalized linear methods and dimension-reduction methods improve performance by controlling the effective number of parameters. Generalized linear models add univariate nonlinearities but do not dominate simpler regularized linear approaches. Trees and neural networks perform best because they can represent interactions among predictors. It converts forecasts into value-weighted decile portfolios and long-short spreads, then evaluates realized returns and Sharpe ratios. This portfolio analysis is the bridge from predictive accuracy to asset-pricing relevance.

The dominant predictors are price trends, liquidity, and volatility: short-term reversal, momentum, industry momentum, market equity, dollar volume, turnover, bid-ask spread, total volatility, idiosyncratic volatility.


# Hypothesis Overview

<!-- Formally detail the paper's key hypotheses. -->
<!-- See @PetersonReplication p. 2 for details. -->

1: Regularized models outperform unrestricted OLS out of sample. Full OLS should perform poorly when all baseline predictors are included, while OLS, elastic net, PCR, and PLS should improve out-of-sample prediction by reducing overfit.

2: Nonlinear interaction models outperform linear methods. Random forests, gradient boosted trees, and neural networks should improve over linear and generalized-linear models because they can capture interactions among stock characteristics and macro states.

3:  Forecast-sorted decile portfolios should show monotonicity in realized returns, and long-short portfolios should earn positive average excess returns and meaningful Sharpe ratios.

4: Important predictors are stable and economically interpretable. Variable importance rankings should emphasize momentum, liquidity, and volatility signals.

# Literature Review

<!-- Write your literature review. See @PetersonReplication p. 2-4 for details. This -->
<!-- section must include paragraphs at least for the 3-5 key references for the -->
<!-- paper to be replicated, similar work, implementation references, more recent -->
<!-- references where available, and any references with attempt to refute the -->
<!-- hypotheses of the replicated work.  A full literature review may contain 20-50 -->
<!-- references.  Not all will be covered in the same level of detail.  Important -->
<!-- references probably warrant an entire paragraph, but similar work can probably -->
<!-- be covered together in 1-2 paragraphs for multiple related works. -->

1. **Kelly, B. T., Pruitt, S., & Su, Y. (2019)**. "Characteristics Are Covariances: A Unified Model of Risk and Return". Journal of Financial Economics, 134(3), 501–524. ([Link](https://doi.org/10.1016/j.jfineco.2019.05.001))


This paper proposes to implement Instrumented Principal Component
Analysis (IPCA), a new latent factor model that allows both factors and their loadings to vary over time as flexible functions of observable firm characteristics, directly linking the characteristics literature to the factor pricing literature. IPCA instruments for unobservable time-varying betas using the same observable characteristics that the cross-sectional literature uses to predict returns, and finds that five latent IPCA factors explain the cross-section of average returns significantly more accurately than existing factor models such as the Fama-French five-factor model. Crucially, the method finds that when risk compensation rather than mispricing drives characteristic-return relationships, the anomaly intercepts are small and statistically insignificant, offering a risk-based interpretation for many widely cited anomalies. This paper is directly connected to the current replication project because IPCA and Gu, Kelly, and Xiu's machine learning framework share a co-author and represent complementary strategies for handling the same high-dimensional characteristic space, with IPCA imposing economic factor structure and the machine learning approach leaving the functional form unrestricted; comparing both approaches deepens the understanding of what drives the predictive gains documented in the main paper.

2. **Freyberger, J., Neuhierl, A., & Weber, M. (2020)**. "Dissecting Characteristics Nonparametrically". The Review of Financial Studies, 33(5), 2326–2377. ([Link](https://doi.org/10.1093/rfs/hhz123))

This paper proposes a nonparametric, penalized regression framework to identify which stock characteristics carry incremental predictive power for the cross-section of expected returns, while allowing their functional form to be flexible rather than restricted to linearity. The authors use the adaptive group LASSO to simultaneously select among a large number of candidate characteristics and estimate their nonparametric effects on expected returns, finding that many previously identified return predictors fail to provide incremental information once others are controlled for, and that nonlinearities are quantitatively important. Their method achieves higher out-of-sample explanatory power compared to linear panel regressions. Published in the same issue of the Review of Financial Studies as Gu, Kelly, and Xiu, this paper is a direct contemporary that also concludes nonlinear functional forms are essential for cross-sectional return prediction, and it is explicitly cited in the main paper as a complementary approach that uses shrinkage rather than tree-based interactions to handle the high-dimensional predictor environment.

3. **Kozak, S., Nagel, S., & Santosh, S. (2020).** "Shrinking the Cross Section". Journal of Financial Economics, 135(2), 271–292. ([Link](https://doi.org/10.1016/j.jfineco.2019.06.008))

This paper constructs a robust stochastic discount factor (SDF) that summarizes the joint explanatory power of a large number of cross-sectional return predictors by applying economically motivated shrinkage to a principal components representation of the candidate factors. The authors find that a characteristics-sparse SDF with only a small number of factors cannot adequately summarize the cross-section of expected stock returns, but that a relatively small number of principal components of the full universe of potential characteristics-based factors can approximate the SDF quite well, achieving robust out-of-sample performance by shrinking the contributions of low-variance components. The paper won the 2020 Fama-DFA Prize for the best paper in the Journal of Financial Economics in asset pricing, and its central finding that a high-dimensional predictor set is necessary to characterize the cross-section aligns directly with Gu, Kelly, and Xiu's evidence that machine learning methods outperform low-dimensional linear models precisely because they can leverage a large, interacting set of predictors.

4. **Jensen, T. I., Kelly, B. T., & Pedersen, L. H. (2023)**. "Is There a Replication Crisis in Finance?" The Journal of Finance, 78(5), 2465–2518. ([Link](https://doi.org/10.1111/jofi.13249))


This paper directly challenges the prevailing pessimism about the replicability of cross-sectional return predictors, using a Bayesian model of factor replication to provide a more optimistic assessment of the literature's collective findings. The authors find that the majority of asset pricing factors can be replicated, cluster into 13 distinct themes that are significant components of the tangency portfolio, hold out-of-sample in a new dataset covering 93 countries, and have evidence that is actually strengthened rather than weakened by the large number of observed factors. The paper develops and estimates a Bayesian model of factor replication that leads to fundamentally different conclusions than the multiple testing critique advanced by Harvey, Liu, and Zhu. This paper is directly relevant to the current project because it validates the premise underlying Gu, Kelly, and Xiu's use of a 900+ predictor set — namely, that a large body of characteristics carries genuine return-predictive information — and because Bryan Kelly is a co-author of both this paper and the main paper, representing the continuation of his research program on machine learning and cross-sectional asset pricing.

5. **Chen, A. Y., & Zimmermann, T. (2022)**. "Open Source Cross-Sectional Asset Pricing". Critical Finance Review, 11(2), ([Link](https://doi.org/10.1561/104.00000112))


This paper provides a comprehensive open-source replication of nearly all cross-sectional stock return predictors published in the academic literature, making the underlying data and code freely available to researchers seeking to work with the full universe of characteristics used in papers like Gu, Kelly, and Xiu. The authors successfully reproduce 319 characteristics drawn from prior meta-studies and find that for the 161 characteristics that were clearly significant in the original papers, 98% of their long-short portfolios produce t-statistics above 1.96. The paper directly engages with the replication crisis debate, noting that while several papers argue much of the cross-sectional predictability literature is false due to reliance on invalid statistical methods, their reproductions suggest a much higher success rate than the pessimistic view implies. For replication purposes, this paper is the most practically important companion to the current project, as it provides the actual cleaned, lagged, and standardized characteristic data needed to reconstruct the 94-variable predictor set used by Gu, Kelly, and Xiu, and its freely accessible codebase significantly reduces the data engineering burden of replicating the main paper from scratch.

# Replication

<!-- Now we move on to the actual replication.  The sections included here are all -->
<!-- necessary, but the may not be sufficient.  Add additional sections and sub-sections -->
<!-- as required to describe your work and make your analytical case. -->

## Data

<!-- Describe the approach that the replication is taking to Data. -->
<!-- See @PetersonReplication p. 4-5 for details. -->
<!-- Describe both the data used in the original paper, and the data you are using -->
<!-- for replication.  For your replicated data, include detailed descriptions of -->
<!-- obtaining, parsing, and cleaning the data to prepare it for use.  Describe data -->
<!-- quality issues. -->

The original paper uses monthly U.S. equity data from CRSP for NYSE, AMEX, and NASDAQ firms from March 1957 to December 2016. The risk-free rate is the Treasury-bill rate. Stock-level predictors consist of 94 firm characteristics based on Green, Hand, and Zhang (2017), plus 74 two-digit SIC industry dummies. Macro predictors follow Welch and Goyal (2008): dividend-price ratio (`dp`), earnings-price ratio (`ep`), book-to-market ratio (`bm`), net equity expansion (`ntis`), Treasury-bill rate (`tbl`), term spread (`tms`), default spread (`dfy`), and stock variance (`svar`).

The data has be organized as follows:

- `data/raw/crsp_monthly_returns.parquet`: monthly stock returns, identifiers, prices, shares, exchange/share codes, SIC, and market equity inputs.
- `data/raw/firm_characteristics.parquet`: stock characteristics with release-date-safe timestamps.
- `data/external/macro_predictors.parquet`: Welch-Goyal-style macro predictors.
- `data/external/risk_free_rate.parquet`: monthly Treasury-bill risk-free rate if not included with macro predictors.
- `data/processed/monthly_panel.parquet`: final one-row-per-stock-month modeling panel.

The final panel must include at minimum `date`, `permno`, `ret_excess_lead1`, and the feature columns. For economic portfolio tests it should also include `market_equity` and `sic2`.

Monthly characteristics should be available by the end of the prediction month, quarterly characteristics should be lagged at least four months, and annual characteristics should be lagged at least six months. Missing characteristic values should be handled cross-sectionally within each month, following the original paper's median-imputation approach before ranking.

## Replication of Key Analytical Techniques

This section states the mathematical specification that the replication should implement. The notation follows Gu, Kelly, and Xiu (2020) unless otherwise noted {cite}`GuKellyXiu2020`.

### 1. Target and information set

For stock $i$ and month $t$, the excess-return model is

$$
r_{i,t+1} = E_t(r_{i,t+1}) + \epsilon_{i,t+1},
$$

with conditional expectation

$$
E_t(r_{i,t+1}) = g^*(z_{i,t}).
$$

The replication target is therefore to estimate a forecasting rule

$$
\hat r_{i,t+1} = \hat g(z_{i,t})
$$

using only information observable by the end of month $t$.

### 2. Feature construction

Let $c_{i,t}$ be the vector of stock characteristics and $x_t$ be the vector of macro predictors plus a constant. The paper's baseline stock-level covariates are

$$
z_{i,t} = x_t \otimes c_{i,t},
$$

where $\otimes$ is the Kronecker product. With 94 characteristics, 8 macro predictors, one constant, and 74 industry dummies, the baseline feature count is

$$
94 \times (8 + 1) + 74 = 920.
$$

Each characteristic is ranked cross-sectionally each month and mapped to `[-1, 1]`. If $rank_{i,t,j}$ is the percentile rank of characteristic $j$, a simple implementation is

$$
\tilde c_{i,t,j} = -1 + 2 \times rank_{i,t,j}.
$$

### 3. Pooled OLS

The simple linear model is

$$
g(z_{i,t};\theta) = z_{i,t}'\theta.
$$

The pooled least-squares objective is

$$
\hat\theta_{OLS} = \arg\min_{\theta}\frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}\left(r_{i,t+1} - z_{i,t}'\theta\right)^2.
$$

This model is included as a benchmark. In the original paper, unrestricted OLS with the full 920-feature set performs poorly because the predictor set is too large and noisy relative to the signal {cite}`GuKellyXiu2020`.

### 4. OLS-3 benchmark

The sparse OLS benchmark uses only three characteristics: size, book-to-market/value, and momentum. Let

$$
z^{(3)}_{i,t} = \left[ size_{i,t}, value_{i,t}, momentum_{i,t} \right].
$$

Then

$$
\hat r_{i,t+1}^{OLS3} = (z^{(3)}_{i,t})'\hat\theta_3.
$$

This benchmark checks whether machine learning improves over a deliberately simple characteristic model.

### 5. Weighted and Huber robust loss

The paper also discusses weighted least squares:

$$
L_W(\theta)=\frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}w_{i,t}\left(r_{i,t+1}-g(z_{i,t};\theta)\right)^2.
$$

For heavy-tailed financial returns, Huber loss replaces squared loss with

$$
H(u;\xi)=
\begin{cases}
u^2, & |u| \leq \xi, \\
2\xi |u| - \xi^2, & |u| > \xi,
\end{cases}
$$

where

$$
u = r_{i,t+1} - g(z_{i,t};\theta).
$$

The robust objective is

$$
L_H(\theta)=\frac{1}{NT}\sum_{i=1}^{N}\sum_{t=1}^{T}H\left(r_{i,t+1}-g(z_{i,t};\theta);\xi\right).
$$

### 6. Elastic net

Elastic net keeps the linear forecast form but regularizes coefficients:

$$
\hat\theta_{EN}=\arg\min_{\theta}\left[L(\theta)+\phi(\theta;\lambda,\rho)\right],
$$

where

$$
\phi(\theta;\lambda,\rho)=\lambda(1-\rho)\sum_{j=1}^{P}|\theta_j| + \frac{\lambda\rho}{2}\sum_{j=1}^{P}\theta_j^2.
$$

The $L_1$ part performs variable selection and the $L_2$ part shrinks correlated predictors. The tuning parameters $\lambda$ and $\rho$ are chosen on the validation sample.

### 7. Principal components regression (PCR)

Stack all stock-month observations into a target vector $R$ and predictor matrix $Z$:

$$
R = Z\theta + E.
$$

PCR replaces $Z$ with $K$ principal components:

$$
R = (Z\Omega_K)\theta_K + \tilde E.
$$

The $j$th principal-component weight vector solves

$$
w_j = \arg\max_w Var(Zw),
$$

subject to

$$
w'w=1, \quad Cov(Zw, Zw_l)=0 \quad \text{for } l<j.
$$

PCR chooses components that preserve predictor covariance, then estimates $\theta_K$ by OLS. The number of components $K$ is validation-tuned.

### 8. Partial least squares (PLS)

PLS also uses

$$
R = (Z\Omega_K)\theta_K + \tilde E,
$$

but chooses components to maximize association with future returns. The $j$th PLS component solves

$$
w_j = \arg\max_w Cov^2(R, Zw),
$$

subject to

$$
w'w=1, \quad Cov(Zw, Zw_l)=0 \quad \text{for } l<j.
$$

This makes PLS more supervised than PCR because return predictability enters the dimension-reduction step {cite}`KellyPruitt2013 KellyPruitt2015`.

### 9. Generalized linear model with spline expansion

The generalized linear model adds nonlinear transformations of each predictor but remains additive across predictors:

$$
g(z;\theta,p)=\sum_{j=1}^{P}p(z_j)'\theta_j.
$$

The paper uses second-order spline basis functions such as

$$
p(z_j)=\left[1, z_j, (z_j-c_1)^2, \ldots, (z_j-c_{K-2})^2\right]'.
$$

To control the enlarged parameter set, the model uses group lasso:

$$
\phi(\theta;\lambda,K)=\lambda\sum_{j=1}^{P}\left(\sum_{k=1}^{K}\theta_{j,k}^2\right)^{1/2}.
$$

This selects or removes all spline terms for a predictor as a group.

### 10. Regression tree

A tree partitions the predictor space into $K$ terminal leaves $C_k(L)$ with maximum depth $L$. The prediction rule is

$$
g(z_{i,t};\theta,K,L)=\sum_{k=1}^{K}\theta_k \mathbf{1}\{z_{i,t}\in C_k(L)\}.
$$

For a leaf $C_k$, the fitted value is the average outcome inside the leaf:

$$
\theta_k = \frac{1}{|C_k|}\sum_{(i,t):z_{i,t}\in C_k}r_{i,t+1}.
$$

A split is chosen to reduce squared-error impurity:

$$
H(\theta,C)=\frac{1}{|C|}\sum_{z_{i,t}\in C}\left(r_{i,t+1}-\theta\right)^2.
$$

A tree of depth $L$ can represent interactions up to roughly order $L-1$, which is why tree models are important for this paper's interaction hypothesis.

### 11. Random forest

A random forest averages predictions from many bootstrapped trees:

$$
\hat g_{RF}(z)=\frac{1}{B}\sum_{b=1}^{B}T_b(z;\hat\theta_b).
$$

Each tree is trained on a bootstrap sample, and each split considers only a random subset of predictors. The main validation-tuned hyperparameters are the number of trees $B$, tree depth $L$, minimum leaf size, and number of candidate split variables.

### 12. Gradient boosted regression trees

Gradient boosting builds an additive ensemble of shallow trees. Initialize with a constant forecast $F_0(z)$. At step $b$, fit a tree $T_b(z)$ to the current residuals or negative gradient, then update

$$
F_b(z)=F_{b-1}(z)+\nu T_b(z),
$$

where $\nu\in(0,1)$ is the learning rate. The final forecast is

$$
\hat g_{GBRT}(z)=F_B(z)=F_0(z)+\nu\sum_{b=1}^{B}T_b(z).
$$

The main tuning parameters are tree depth $L$, learning rate $\nu$, and number of boosting iterations $B$.

### 13. Feed-forward neural networks

The neural-network input layer is

$$
x^{(0)} = [1,z_1,\ldots,z_P]'.
$$

For hidden layer $l$, neuron $k$ is computed recursively as

$$
x_k^{(l)} = ReLU\left((x^{(l-1)})'\theta_k^{(l-1)}\right),
$$

with

$$
ReLU(a)=\max(0,a).
$$

The final output is

$$
g(z;\theta)= (x^{(L-1)})'\theta^{(L-1)}.
$$

The paper studies five architectures:

$$
NN1=[32],\quad NN2=[32,16],\quad NN3=[32,16,8],
$$

$$
NN4=[32,16,8,4],\quad NN5=[32,16,8,4,2].
$$

Training minimizes penalized prediction loss and uses regularization devices such as early stopping, learning-rate shrinkage, batch normalization, and ensembling across random seeds {cite}`GuKellyXiu2020 ChenPelgerZhu2019`.

### 14. Validation and testing protocol

For each rolling split, the workflow is

$$
\theta(\eta) = \arg\min_{\theta}L_{train}(\theta;\eta),
$$

where $\eta$ denotes hyperparameters. Choose

$$
\hat\eta = \arg\min_{\eta}L_{validation}(\theta(\eta);\eta).
$$

Then evaluate only on the test set:

$$
\hat r_{i,t+1}=g(z_{i,t};\hat\theta(\hat\eta)), \quad (i,t)\in T_{test}.
$$

The original design begins with 18 training years, 12 validation years, and then annual rolling or expanding test evaluation.

### 15. Forecast accuracy

The main panel forecast metric is out-of-sample $R^2$ against a zero forecast:

$$
R^2_{oos}=1-\frac{\sum_{(i,t)\in T_3}(r_{i,t+1}-\hat r_{i,t+1})^2}{\sum_{(i,t)\in T_3}r_{i,t+1}^2}.
$$

The denominator is not demeaned because the benchmark is the zero excess-return forecast, not the historical mean.

### 16. Pairwise model comparison

For two models A and B, define the cross-sectional average squared-error difference each month:

$$
d_{AB,t+1}=\frac{1}{n_t}\sum_i\left(e^2_{A,i,t+1}-e^2_{B,i,t+1}\right),
$$

where

$$
e_{A,i,t+1}=r_{i,t+1}-\hat r^A_{i,t+1}.
$$

The Diebold-Mariano-style statistic is

$$
DM_{AB}=\frac{\bar d_{AB}}{\widehat{se}(\bar d_{AB})},
$$

with Newey-West standard errors for $\bar d_{AB}$ {cite}`DieboldMariano1995 GuKellyXiu2020`.

### 17. Variable importance

The paper's first importance measure is the reduction in out-of-sample $R^2$ after setting predictor $j$ to zero:

$$
VI_j = R^2_{oos} - R^2_{oos,-j}.
$$

For differentiable models, a derivative-based sensitivity measure is

$$
SSD_j = \sum_{(i,t)\in T}\left(\frac{\partial g(z;\theta)}{\partial z_j}\bigg|_{z=z_{i,t}}\right)^2.
$$

Tree models can instead report mean decrease in impurity or permutation importance.

### 18. Portfolio sorts

Each month, sort stocks into deciles by forecast $\hat r_{i,t+1}$. For decile $q$, the value-weighted return is

$$
R_{q,t+1}=\sum_{i\in q_t}w_{i,t}r_{i,t+1},
$$

where

$$
w_{i,t}=\frac{ME_{i,t}}{\sum_{j\in q_t}ME_{j,t}}.
$$

The long-short spread is

$$
R^{LS}_{t+1}=R_{10,t+1}-R_{1,t+1}.
$$

Annualized Sharpe ratio is

$$
SR_{ann}=\sqrt{12}\frac{\bar R^{LS}}{\sigma(R^{LS})}.
$$

These portfolio equations turn the statistical forecast into the economic test of whether the model produces useful expected-return rankings.


## Hypothesis Tests

<!-- After replicating the initial work, it is time to evaluate the hypotheses of -->
<!-- the replicated work. Those hypotheses were identified above, before you started -->
<!-- replication. Describe, in detail, the statistical tests you perform to refute or -->
<!-- validate the hypotheses in the replicated work.  This should go beyond any explicit -->
<!-- tests performed in the original paper. -->

The core forecast test is panel out-of-sample $R^2$ against a zero benchmark:

$$
R^2_{oos} = 1 - \frac{\sum_{(i,t) \in T_3} (r_{i,t+1} - \hat r_{i,t+1})^2}{\sum_{(i,t) \in T_3} r_{i,t+1}^2}.
$$

A positive value means the model beats the naive forecast that every stock's future excess return is zero. This benchmark is intentionally stricter than historical-mean forecasts for individual stocks.

Pairwise model comparisons should use a Diebold-Mariano-style statistic on cross-sectional average squared-error differences. For models A and B, compute each month:

$$
d_{AB,t+1} = \frac{1}{n_t}\sum_i \left(e^2_{A,i,t+1} - e^2_{B,i,t+1}\right).
$$

Then test whether the time-series mean of `d` differs from zero, using Newey-West standard errors. A positive value means model B has lower squared error than model A under this sign convention.

The economic tests sort stocks into deciles by forecast each month, compute value-weighted decile returns, and report the high-minus-low spread. The main outputs are average monthly return, annualized Sharpe ratio, monotonicity across deciles, and performance for large-stock and small-stock subsamples.


## Analytical Process



After the model-ready panel exists, the analysis should run in the following order.

### Analysis 1: Descriptive data checks

Produce a table with:

- number of stock-month observations,
- number of unique stocks,
- sample start and end dates,
- average stocks per month,
- missing-value rate by characteristic family,
- distribution of `ret_excess_lead1`,
- distribution of `market_equity`,
- number of industries each month.

These checks verify that the replication sample resembles the original paper's broad CRSP universe.

### Analysis 2: Rolling train-validation-test splits

Generate time-ordered splits. The starting design is 18 training years, 12 validation years, and 1 test year per refit:

$$
T_1 = \text{training},\quad T_2 = \text{validation},\quad T_3 = \text{testing}.
$$

For each split, estimate model parameters on $T_1$, tune hyperparameters on $T_2$, and store forecasts for $T_3$ only.

### Analysis 3: Model-level forecast tables

For each model, store a prediction panel with columns:

```text
date, permno, ret_excess_lead1, forecast, model, market_equity, sic2
```

Then compute:

$$
R^2_{oos}=1-\frac{\sum(r-\hat r)^2}{\sum r^2}.
$$

Report the main table by model: `OLS`, `OLS-3`, `ENet`, `PCR`, `PLS`, `GLM`, `RF`, `GBRT`, `NN1`-`NN5`.

### Analysis 4: Pairwise model comparison

For each model pair A and B, compute monthly cross-sectional average error differences:

$$
d_{AB,t+1}=\frac{1}{n_t}\sum_i(e^2_{A,i,t+1}-e^2_{B,i,t+1}).
$$

Report a Diebold-Mariano-style statistic:

$$
DM_{AB}=\frac{\bar d_{AB}}{\widehat{se}(\bar d_{AB})}.
$$

This produces a matrix like the paper's pairwise model comparison table.

### Analysis 5: Portfolio sorts

For every month and model:

1. Sort stocks into deciles by forecast.
2. Compute value-weighted return for each decile.
3. Compute high-minus-low spread:

$$
R^{LS}_{t+1}=R_{10,t+1}-R_{1,t+1}.
$$

4. Report mean monthly return, annualized volatility, annualized Sharpe, and monotonicity across deciles.

### Analysis 6: Subsample analysis

Repeat forecast and portfolio analysis for:

- top 1,000 stocks by market equity,
- bottom 1,000 stocks by market equity,
- pre-2000 vs post-2000 periods,
- high-volatility vs low-volatility macro regimes,
- with and without financial firms if the required identifiers are available.

### Analysis 7: Variable importance

For each trained model and feature group, report importance using:

$$
VI_j = R^2_{oos} - R^2_{oos,-j}.
$$

Group importance should also be computed for:

- momentum and reversal,
- liquidity,
- volatility and beta,
- valuation,
- profitability and investment,
- macro interactions,
- industry dummies.

### Analysis 8: Figures and report artifacts

Save outputs under `reports/`:

| Output | Location |
|---|---|
| Sample summary | `reports/tables/sample_summary.csv` |
| Model R2 table | `reports/tables/model_r2.csv` |
| Pairwise DM table | `reports/tables/model_dm_tests.csv` |
| Decile portfolio returns | `reports/tables/decile_portfolios.csv` |
| Long-short performance | `reports/tables/long_short_performance.csv` |
| Variable importance | `reports/tables/variable_importance.csv` |
| R2 bar chart | `reports/figures/model_r2.png` |
| Decile return plot | `reports/figures/decile_returns.png` |
| Variable importance plot | `reports/figures/variable_importance.png` |


## Overfitting

The original paper is explicitly designed around overfitting control. It separates training, validation, and testing periods; tunes hyperparameters only on validation data; benchmarks individual-stock forecasts against zero; and uses multiple model classes to check whether results are method-specific.

Several overfitting risks remain important for replication. The first is data construction. If accounting variables are not lagged conservatively, the model can use information unavailable at the forecast date. The second is model-selection leakage. If the researcher repeatedly revises features or hyperparameter grids after looking at test performance, the test set becomes contaminated. The third is survivorship or universe bias from using only stocks with complete data or imposing filters that differ from the paper.

# Future Work

<!-- What additional work on this topic should be performed in the future, if this -->
<!-- project were to be picked up again or continued? -->

# Conclusions

<!-- Summarize the project and describe your conclusions.  This sections can -->
<!-- range from 1-2 paragraphs to 1-2 pages. -->

\newpage

![CC-BY](cc_by_88x31.png)

# References

```{bibliography}
```

